# SparseDict Phase 4: DIV2K micro-overfit and inference

Run this notebook with a Colab GPU runtime for the 200-epoch experiment. The DIV2K archive is about 3.28 GiB; this notebook extracts only the requested training images. DIV2K is provided for academic research only, and image copyrights remain with their original owners. Run the download cell only if your use complies with those terms.

The runtime filesystem is temporary. The final cell packages checkpoints, metrics, and restored images for download.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/kalpeshjp89/SparseDict.git"
REPO_DIR = Path("/content/SparseDict")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
%cd /content/SparseDict
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun this cell.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
DATA_ROOT = Path("/content/DIV2K")
subprocess.run(
    [sys.executable, "-m", "scripts.download_div2k", "--output-dir", str(DATA_ROOT),
     "--image-count", "10", "--accept-license"],
    check=True,
)

In [ ]:
DEGRADATION = "noise"  # Options: noise, super_resolution, rain
OUTPUT_ROOT = Path("/content/phase4_output")
TRAIN_OUTPUT = OUTPUT_ROOT / DEGRADATION
train_result = subprocess.run(
    [sys.executable, "-m", "experiments.run_phase4",
     "--data-root", str(DATA_ROOT), "--degradation", DEGRADATION,
     "--image-count", "10", "--epochs", "200",
     "--output-dir", str(TRAIN_OUTPUT), "--device", "cuda"],
    check=False,
)
metrics_path = TRAIN_OUTPUT / "metrics.json"
if metrics_path.exists():
    import json
    metrics = json.loads(metrics_path.read_text())
    print("Final validation PSNR:", metrics["final_validation_psnr"], "dB")
    print("Meets 29.5 dB target:", metrics["pass"])
if train_result.returncode != 0:
    print("Training command returned", train_result.returncode,
          "(the script returns nonzero if the PSNR target was not reached).")
if not (TRAIN_OUTPUT / "moe_topk_restorer.pt").exists():
    raise RuntimeError("Training did not produce a checkpoint; inspect the cell output.")

In [ ]:
INFERENCE_OUTPUT = TRAIN_OUTPUT / "inference"
subprocess.run(
    [sys.executable, "-m", "experiments.infer_phase4",
     "--data-root", str(DATA_ROOT),
     "--checkpoint", str(TRAIN_OUTPUT / "moe_topk_restorer.pt"),
     "--image-count", "10", "--device", "cuda",
     "--output-dir", str(INFERENCE_OUTPUT)],
    check=True,
)
inference_metrics = json.loads((INFERENCE_OUTPUT / "inference_metrics.json").read_text())
print("Mean full-image tiled inference PSNR:", inference_metrics["mean_psnr"], "dB")

In [ ]:
import shutil
from google.colab import files

archive_path = Path("/content/phase4_results.zip")
shutil.make_archive(str(archive_path.with_suffix("")), "zip", OUTPUT_ROOT)
files.download(str(archive_path))